In [ ]:
import warnings; warnings.simplefilter('ignore')
import warnings; warnings.simplefilter('ignore')

import hotspot
import scanpy as sc
import muon as mu
import anndata
import numpy as np
import mplscience
adata = anndata.read_h5ad('./hippo/hippo.h5ad')
# 确保所有基因名称都是唯一的
adata.var_names_make_unique()

# 移除少于200个基因的细胞
sc.pp.filter_cells(adata, min_genes=200)

# 移除在少于3个细胞中表达的基因
sc.pp.filter_genes(adata, min_cells=3)

# 展示在每个细胞中产生最高计数的基因
sc.pl.highest_expr_genes(adata, n_top=20)
# 将线粒体基因组标注为'mt'
adata.var['mt'] = adata.var_names.str.startswith('MT-') 
 
# 将核糖体蛋白组标注为'rb'
adata.var['rb'] = adata.var_names.str.startswith(('RPS','RPL')) 

# 计算
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
sc.pp.calculate_qc_metrics(adata, qc_vars=['rb'], percent_top=None, log1p=False, inplace=True)
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt', 'pct_counts_rb'], 
             jitter=0.4, multi_panel=True)
with mplscience.style_context():
    sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
                jitter=0.4, multi_panel=True, log=True)

mdata = adata
#计算PCA
mdata.layers["counts"] = mdata.X.copy()
sc.pp.normalize_total(mdata)
sc.pp.log1p(mdata)
mdata.layers["log_normalized"] = mdata.X.copy()
sc.pp.scale(mdata)
sc.tl.pca(mdata)
with mplscience.style_context():sc.pl.pca_variance_ratio(mdata)
mdata.layers["counts_csc"] = mdata.layers["counts"].tocsc()
hs = hotspot.Hotspot(mdata,layer_key="counts_csc",model='danb',latent_obsm_key="X_pca",umi_counts_obs_key="total_counts")

hs.create_knn_graph(
  weighted_graph=False, n_neighbors=30,
)

####计算基因自相关性
#hs_results = hs.compute_autocorrelations(jobs=4)#报错了，就不多线程了
hs_results = hs.compute_autocorrelations()
hs_results.head(15)
# Select the genes with significant lineage autocorrelation
hs_genes = hs_results.loc[hs_results.FDR < 0.05].sort_values('Z', ascending=False).head(500).index

# Compute pair-wise local correlations between these genes
#lcz = hs.compute_local_correlations(hs_genes, jobs=4)
lcz = hs.compute_local_correlations(hs_genes)

modules = hs.create_modules(
  min_gene_threshold=15, core_only=True, fdr_threshold=0.05
)

modules.value_counts()
hs.plot_local_correlations(vmin=-12, vmax=12).savefig("./hippo/plot_local_correlations.pdf")
# Select the genes with significant lineage autocorrelation
hs_genes = hs_results.loc[hs_results.FDR < 0.05].sort_values('Z', ascending=False).head(500).index

# Compute pair-wise local correlations between these genes
#lcz = hs.compute_local_correlations(hs_genes, jobs=4)
lcz = hs.compute_local_correlations(hs_genes)

modules = hs.create_modules(
  min_gene_threshold=15, core_only=True, fdr_threshold=0.05
)

modules.value_counts()
hs.plot_local_correlations(vmin=-12, vmax=12)
a = hs.plot_local_correlations(vmin=-12, vmax=12)
# Show the top genes for a module
module = 9
results = hs.results.join(hs.modules)
results = results.loc[results.Module == module]
results.sort_values('Z', ascending=False).head(10)
#Summary Module Scores
module_scores = hs.calculate_module_scores()
module_scores.head()
sc.pp.neighbors(mdata)
sc.tl.umap(mdata)
sc.pl.umap(mdata, frameon=False)

module_cols = []
for c in module_scores.columns:
  key = f"Module {c}"
mdata.obs[key] = module_scores[c]
module_cols.append(key)

with mplscience.style_context():
  sc.pl.umap(mdata, color=module_cols, frameon=False, vmin=-1, vmax=1)

lcz.to_csv('./hippo/hippo_genemodule_cor.csv', index=False)
results = hs.results.join(hs.modules)
results.to_csv('./hippo/hippo_genemodule.csv', index=True)